In [13]:
import codecs, re, torch, gc, json
from torch import device
from transformers import DistilBertForSequenceClassification
from torch.nn import CrossEntropyLoss
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score, precision_score, recall_score, accuracy_score
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


loading dataset

In [2]:
# loading functions for the dataset
def load_movies(path2data): # 1 classe par répertoire
    alltxts = [] # init vide
    labs = []
    cpt = 0
    for cl in os.listdir(path2data): # parcours des fichiers d'un répertoire
        for f in os.listdir(path2data+cl):
            txt = open(path2data+cl+'/'+f).read()
            alltxts.append(txt)
            labs.append(cpt)
        cpt+=1 # chg répertoire = cht classe

    return alltxts,labs

def load_movies_test(path2data):
    alltxts = []
    c = 0
    with open(path2data, 'r') as file:
        for line in file:        
            alltxts.append(line)
    return alltxts

# load dataset train
path = "./Dataset/movies1000/"
alltxt,alllabs = load_movies(path)
alltxt, alllabs = np.array(alltxt), np.array(alllabs)

# load dataset test
alltxt_test = load_movies_test("Test_set/testSentiment.txt")
alltxt_test = np.array(alltxt_test)

In [3]:
len(alltxt[0]),alllabs[0],alltxt_test[0]

(4430,
 np.int64(0),
 np.str_("Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.\n"))

### v1: 
maxL = 256

configs = [
        {"name": "config1_all_frozen", "freeze_all": True, "lr": 5e-6, "max_epochs": 8},
        {"name": "config2_last2_layers", "freeze_all": False, "unfreeze_last": 2, "lr": 1e-5, "dropout_config": {"dropout": 0.3, "attention_dropout": 0.3, "seq_classif_dropout": 0.5}},
        {"name": "config3_last1_layer", "freeze_all": False, "unfreeze_last": 1, "lr": 3e-6, "max_epochs": 10}
    ]

### v2:
microsoft/deberta-v3-base

maxL = 256

In [4]:
version = 2

tokenizing

In [5]:
# tokenization
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
"""
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')

#from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
#tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased') # smaller

# tokenize train set
import torch
maxL = 256 # Max length of the sequence #512

inputs_ids_train = []
attention_masks_train = []

for i in range(len(alltxt)):
    if(i%2500==0):print(i)
    string_tokenized = tokenizer(alltxt[i], return_tensors="pt",
                                        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
                                        max_length=maxL,  # set max length
                                        truncation=True,  # truncate longer messages
                                        #pad_to_max_length=True
                                        padding='max_length',  # add padding
                                        return_attention_mask=True)

    inputs_ids_train.append(string_tokenized['input_ids'])
    attention_masks_train.append(string_tokenized['attention_mask'])

# convert into one big tensor
inputs_ids_train = torch.cat(inputs_ids_train, dim=0)
attention_masks_train = torch.cat(attention_masks_train, dim=0)


## tokenize test set
inputs_tokens_test = []
attention_masks_test = []
for i in range(len(alltxt_test)):
    if(i%2500==0): print(i)
    string_tokenized = tokenizer(alltxt_test[i], return_tensors="pt",
                                        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
                                        max_length=maxL,  # set max length
                                        truncation=True,  # truncate longer messages
                                        #pad_to_max_length=True
                                        padding='max_length',  # add padding
                                        return_attention_mask=True)

    inputs_tokens_test.append(string_tokenized['input_ids'])
    attention_masks_test.append(string_tokenized['attention_mask'])

# convert into one big tensor
input_ids_test = torch.cat(inputs_tokens_test, dim=0)
attention_masks_test = torch.cat(attention_masks_test, dim=0)

torch.save({"input_ids": inputs_ids_train, "attention_masks": attention_masks_train}, f"tokens/movies_train_tokens_dist_{version}.pt")

torch.save({"input_ids": input_ids_test,"attention_masks": attention_masks_test}, f"tokens/movies_pred_test_tokens_dist_{version}.pt")

"""

# if already tokenized: loading train and validation
data = torch.load(f"tokens/movies_train_tokens_dist_{version}.pt")
inputs_ids_train = data["input_ids"]
attention_masks_train = data["attention_masks"]

data = torch.load(f"tokens/movies_pred_test_tokens_dist_{version}.pt")
input_ids_test = data["input_ids"]
attention_masks_test = data["attention_masks"]

y_train_tens = torch.tensor(alllabs, dtype=torch.long)

In [14]:
# pre emptying cache
gc.collect()
torch.cuda.empty_cache()

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)


device: cuda


## cross validation train val no test 

In [8]:
#model = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-base', ignore_mismatched_sizes=True,cache_dir="./deberta_dist_cache").to(dev)
#model = AutoModelForSequenceClassification.from_pretrained("deberta_dist_cache/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3", ignore_mismatched_sizes=True).to(dev)


In [15]:
#from transformers import DistilBertConfig

from networkx import config
from torch.optim.lr_scheduler import ReduceLROnPlateau


def train_fold(config_name, class_weights, train_dataloader, val_dataloader, fold_idx, cv_results, unfreeze_last=1, lr=5e-5, dropout_config=None, max_epochs=15):
    """training function for different configs"""
    torch.backends.cuda.matmul.allow_tf32 = True
    
    if dropout_config:
        cfg = AutoConfig.from_pretrained('microsoft/deberta-v3-base', num_labels=2, **dropout_config)

        #cfg = DistilBertConfig.from_pretrained('distilbert-base-cased', **dropout_config)
    else:
        cfg = AutoConfig.from_pretrained('microsoft/deberta-v3-base', 
            num_labels=2, hidden_dropout_prob=0.2, attention_probs_dropout_prob=0.2, classifier_dropout=0.4)

        #cfg = DistilBertConfig.from_pretrained('distilbert-base-cased', 
            #dropout=0.1, attention_dropout=0.1, seq_classif_dropout=0.3)
    
    model = AutoModelForSequenceClassification.from_pretrained("deberta_dist_cache/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3", config=cfg, ignore_mismatched_sizes=True).to(dev)


    #model = DistilBertForSequenceClassification.from_pretrained(
    #    "bert_dist_cache/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0", 
    #    config=cfg, ignore_mismatched_sizes=True,output_loading_info=False
    #).to(dev)
    
    # freezing logic
    #for param in model.distilbert.parameters():
    #    param.requires_grad = False
    
    #for layer in model.distilbert.transformer.layer[-unfreeze_last:]:
    #    for param in layer.parameters():
    #        param.requires_grad = True

    # Freeze all except specified layers
    for param in model.deberta.parameters():
        param.requires_grad = False

    if unfreeze_last > 0:
        for layer in model.deberta.encoder.layer[-unfreeze_last:]:
            for param in layer.parameters():
                param.requires_grad = True


    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)#, weight_decay=0.01)
    model = model.float()  
    #scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"{config_name}: Trainable {trainable_params/1e6:.1f}M / {total_params/1e6:.1f}M ({100*trainable_params/total_params:.1f}%)")


    #optimizer = optim.Adam(model.parameters(), lr=lr)
    #loss_fn = CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)  # add label smoothing
    loss_fn = CrossEntropyLoss()
    best_val_f1 = 0
    patience_counter = 0
    fold_epochs_evolution = {}
    print(f"----------------{config_name} training-------------------")

    for epoch in range(max_epochs):
        model.train()
        total_loss = 0
        all_train_preds = []
        all_train_labels = []
        all_train_logits = []
        
        accumulation_steps = 2
        optimizer.zero_grad()
        for i, batch in enumerate(train_dataloader):
            b_input_ids, b_attention_mask, b_labels = [t.to(dev) for t in batch]
            model.zero_grad()
            
            outputs = model(
                input_ids=b_input_ids,
                attention_mask=b_attention_mask,
            )
            logits = outputs.logits
            loss = loss_fn(logits, b_labels)
            loss = loss / accumulation_steps
            total_loss += loss.item()
            loss.backward()
            if (i + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            preds = torch.argmax(logits, dim=1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(b_labels.cpu().numpy())
            all_train_logits.append(logits.detach().cpu())
            
            del b_input_ids, b_attention_mask, b_labels
            del outputs, loss, logits, preds
            torch.cuda.empty_cache()
            gc.collect()
        
        avg_train_loss = total_loss / len(train_dataloader)
        
        # Training metrics
        train_f1        = f1_score(all_train_labels, all_train_preds)
        train_precision = precision_score(all_train_labels, all_train_preds)
        train_recall    = recall_score(all_train_labels, all_train_preds)
        train_acc       = accuracy_score(all_train_labels, all_train_preds)
        all_logits_tensor = torch.cat(all_train_logits, dim=0)
        train_probs = torch.softmax(all_logits_tensor, dim=1)[:, 1].numpy()
        
        # ── Validation metrics ───────────────────────────────────────────────
        model.eval()
        all_val_preds = []
        all_val_labels = []
        all_val_logits = []
        
        with torch.no_grad():
            for batch in val_dataloader:
                b_input_ids, b_attention_mask, b_labels = [t.to(dev) for t in batch]
                
                outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
                logits = outputs.logits
                
                preds = torch.argmax(logits, dim=1)
                all_val_preds.extend(preds.cpu().numpy())
                all_val_labels.extend(b_labels.cpu().numpy())
                all_val_logits.append(logits.detach().cpu())
                
                del b_input_ids, b_attention_mask, b_labels
                del outputs, logits, preds
                torch.cuda.empty_cache()
                gc.collect()
        
        # Validation metrics
        val_f1        = f1_score(all_val_labels, all_val_preds)
        val_precision = precision_score(all_val_labels, all_val_preds)
        val_recall    = recall_score(all_val_labels, all_val_preds)
        val_acc       = accuracy_score(all_val_labels, all_val_preds)
        all_val_logits_tensor = torch.cat(all_val_logits, dim=0)
        val_probs = torch.softmax(all_val_logits_tensor, dim=1)[:, 1].numpy()
        
        #scheduler.step(val_f1)
        print(f"Epoch {epoch+1}/{max_epochs}")
        print(f"  Train - Loss: {avg_train_loss:.4f} | F1: {train_f1:.4f} | Precision: {train_precision:.4f} | Recall: {train_recall:.4f} | Accuracy: {train_acc:.4f}")
        print(f"  Val   - F1: {val_f1:.4f} | Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | Accuracy: {val_acc:.4f}")
                
        # Early stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 2:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        fold_epochs_evolution[f"epoch_{epoch+1}"] = {
            "train_loss": avg_train_loss,
            "train_f1": train_f1,
            "train_precision": train_precision,
            "train_recall": train_recall,
            "train_accuracy": train_acc,
            "val_f1": val_f1,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_accuracy": val_acc,
        }
        
        del all_train_logits, all_logits_tensor, train_probs
        del all_val_logits, all_val_logits_tensor, val_probs
        del all_train_preds, all_train_labels, all_val_preds, all_val_labels
        torch.cuda.empty_cache()
        gc.collect()
        
    del model, optimizer, loss_fn
    cv_results[f"fold_{fold_idx}_{config_name}"] = fold_epochs_evolution
    return best_val_f1


In [16]:
n_splits = 3
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
cv_results = {}
fold_idx = 0

for train_idx, val_idx in skf.split(inputs_ids_train, alllabs):
    fold_idx += 1
    print(f"\n{'='*80}")
    print(f"FOLD {fold_idx}/{n_splits}")
    print(f"{'='*80}")
    
    # Split data for this fold
    X_train_fold = inputs_ids_train[train_idx]
    mask_train_fold = attention_masks_train[train_idx]
    y_train_fold = y_train_tens[train_idx]
    
    X_val_fold = inputs_ids_train[val_idx]
    mask_val_fold = attention_masks_train[val_idx]
    y_val_fold = y_train_tens[val_idx]
    

    # Compute weights for weighted sampler
    _, counts = np.unique(alllabs[train_idx], return_counts=True)
    weights = 1/counts
    sample_weights = [weights[label] for label in alllabs[train_idx]]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(alllabs[train_idx]), replacement=True)
    class_weights = torch.tensor(weights, dtype=torch.float).to(dev)

    
    # Create dataloaders
    dataset_train = TensorDataset(X_train_fold, mask_train_fold, y_train_fold)
    dataset_val = TensorDataset(X_val_fold, mask_val_fold, y_val_fold)
    
    train_dataloader = DataLoader(dataset_train, batch_size=8, sampler=sampler)
    val_dataloader = DataLoader(dataset_val, batch_size=8, shuffle=False)
    
    configs = [
    {"config_name": "fix1_classifier_only", "unfreeze_last": 0, "lr": 3e-5},  # ONLY classifier
    {"config_name": "fix2_last4_layers", "unfreeze_last": 4, "lr": 8e-6},
    ]


    for config in configs:
        best_f1 = train_fold(class_weights=class_weights, train_dataloader=train_dataloader, val_dataloader=val_dataloader,fold_idx=fold_idx,cv_results=cv_results, **config)
        print(f"{config['config_name']}: Best val F1 = {best_f1:.4f}")

        torch.cuda.empty_cache()
        gc.collect()


    del dataset_train, dataset_val, train_dataloader, val_dataloader
    torch.cuda.empty_cache()
    gc.collect()

# Save cross-validation results
json.dump(cv_results, open(f"cv_results{version}.json", "w"), indent=2)
print(f"\n{'='*80}")
print(f"Cross-validation complete! Results saved to cv_results{version}.json")
print(f"{'='*80}")



FOLD 1/3


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 31537.32it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: deberta_dist_cache/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MI

OutOfMemoryError: CUDA out of memory. Tried to allocate 376.00 MiB. GPU 0 has a total capacity of 5.61 GiB of which 220.31 MiB is free. Including non-PyTorch memory, this process has 4.93 GiB memory in use. Of the allocated memory 4.56 GiB is allocated by PyTorch, and 285.33 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## train test normal split 

In [ ]:

# ------- weights
val, counts = np.unique(alllabs, return_counts = True)
weights = 1/counts # weights for loss
y_train_tens = torch.tensor(alllabs)

# dataset loading
sample_weights = [weights[label] for label in alllabs]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(alllabs), replacement=True)

dataset_train = TensorDataset(inputs_ids_train,  attention_masks_train, y_train_tens)
dataset_test = TensorDataset(input_ids_test,  attention_masks_test)

train_dataloader = DataLoader(dataset_train, batch_size=16, sampler=sampler)

test_dataloader = DataLoader(dataset_test,batch_size=16,shuffle=False)


In [17]:

# loading not trained model
from transformers import DistilBertConfig, DistilBertForSequenceClassification

config = DistilBertConfig.from_pretrained('distilbert-base-uncased',
    dropout=0.2,
    attention_dropout=0.2,
    seq_classif_dropout=0.4
)

model = DistilBertForSequenceClassification.from_pretrained("bert_dist_cache/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0", config = config,ignore_mismatched_sizes=True)


# Freeze all DistilBERT layers, train only the classifier head
for param in model.distilbert.parameters():
    param.requires_grad = False

# Or unfreeze only the last N encoder layers
for layer in model.distilbert.transformer.layer[-4:]:  # last 4 layers
    for param in layer.parameters():
        param.requires_grad = True


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8504.44it/s]
DistilBertForSequenceClassification LOAD REPORT from: bert_dist_cache/models--distilbert-base-cased/snapshots/6ea81172465e8b0ad3fddeed32b986cdcdcffcf0
Key                                          | Status     |                                                                                               
---------------------------------------------+------------+-----------------------------------------------------------------------------------------------
vocab_transform.weight                       | UNEXPECTED |                                                                                               
vocab_transform.bias                         | UNEXPECTED |                                                                                               
vocab_projector.bias                         | UNEXPECTED |                                                                                               
vocab

In [ ]:

# pre emtpying cache
gc.collect()
torch.cuda.empty_cache()

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(dev)
print("dev", dev)

class_weights = torch.tensor(weights, dtype=torch.float).to(dev)
loss_fn = CrossEntropyLoss()

# ------ running loop

optimizer = optim.Adam(model.parameters(), lr=2e-5)
epochs = 5 # to test

model.to(dev)
epochs_evolution = {}

# ── Training ─────────────────────────────────────────────────────────────────
for epoch in range(epochs):
    model.train()
    total_loss = 0
    all_train_preds = []
    all_train_labels = []
    all_train_logits = []

    for i, batch in enumerate(train_dataloader):
        print("batch", i)
        b_input_ids, b_attention_mask, b_labels = [t.to(dev) for t in batch]
        model.zero_grad()

        outputs = model(
            input_ids=b_input_ids,
            attention_mask=b_attention_mask,
            # labels=b_labels
        )
        # loss = outputs.loss
        logits = outputs.logits
        loss = loss_fn(logits, b_labels)

        total_loss += loss.item()
        loss.backward()
        optimizer.step()

        preds = torch.argmax(logits, dim=1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(b_labels.cpu().numpy())
        all_train_logits.append(logits.detach().cpu())

        del b_input_ids, b_attention_mask, b_labels
        del outputs, loss, logits, preds
        torch.cuda.empty_cache()
        gc.collect()

    avg_train_loss = total_loss / len(train_dataloader)

    # Training metrics
    train_f1        = f1_score(all_train_labels, all_train_preds, average="macro")
    train_precision = precision_score(all_train_labels, all_train_preds, average="macro")
    train_recall    = recall_score(all_train_labels, all_train_preds, average="macro")

    all_logits_tensor = torch.cat(all_train_logits, dim=0)
    train_probs = torch.softmax(all_logits_tensor, dim=1)[:, 1].numpy()
    train_auc = roc_auc_score(all_train_labels, train_probs)
    train_ap  = average_precision_score(all_train_labels, train_probs)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_train_loss:.4f} | "
          f"F1: {train_f1:.4f} | Precision: {train_precision:.4f} | Recall: {train_recall:.4f} | "
          f"AUC: {train_auc:.4f} | AP: {train_ap:.4f}")

    epochs_evolution[f"epoch_{epoch+1}"] = {   # fixed key to track each epoch
        "loss": avg_train_loss,
        "f1": train_f1,
        "precision": train_precision,
        "recall": train_recall,
        "auc": train_auc,
        "ap": train_ap
    }

    del all_train_logits, all_logits_tensor, train_probs
    del all_train_preds, all_train_labels
    torch.cuda.empty_cache()
    gc.collect()

json.dump(epochs_evolution, open("train_res.json", "w"))
print("Training metrics saved to train_res.json")

# ── Prediction on unlabeled test set ─────────────────────────────────────────
model.eval()
all_preds = []
all_probs = []

with torch.no_grad():
    for batch in test_dataloader:
        b_input_ids, b_attention_mask = [t.to(dev) for t in batch]  # no labels

        outputs = model(input_ids=b_input_ids, attention_mask=b_attention_mask)
        logits  = outputs.logits
        probs   = torch.softmax(logits, dim=1)[:, 1]
        preds   = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

        del b_input_ids, b_attention_mask
        del outputs, logits, probs, preds
        torch.cuda.empty_cache()
        gc.collect()

# Save predictions and probabilities
np.save("PRED_test_preds.npy", np.array(all_preds))
np.save("PRED_test_probs.npy", np.array(all_probs))
print(f"Predictions saved — {len(all_preds)} samples")
print(f"Class distribution: { {v: int((np.array(all_preds)==v).sum()) for v in np.unique(all_preds)} }")

print("predictions unique", np.unique(all_preds, return_counts=True))

# ── Final cleanup ─────────────────────────────────────────────────────────────
del all_preds, all_probs
torch.cuda.empty_cache()
gc.collect()

dev cuda
batch 0
batch 1
batch 2
batch 3
batch 4
batch 5
batch 6
batch 7
batch 8
batch 9
batch 10
batch 11
batch 12
batch 13
batch 14
batch 15
batch 16
batch 17
batch 18
batch 19
batch 20
batch 21
batch 22
batch 23
batch 24
batch 25
batch 26
batch 27
batch 28
batch 29
batch 30
batch 31
batch 32
batch 33
batch 34
batch 35
batch 36
batch 37
batch 38
batch 39
batch 40
batch 41
batch 42
batch 43
batch 44
batch 45
batch 46
batch 47
batch 48
batch 49
batch 50
batch 51
batch 52
batch 53
batch 54
batch 55
batch 56
batch 57
batch 58
batch 59
batch 60
batch 61
batch 62
batch 63
batch 64
batch 65
batch 66
batch 67
batch 68
batch 69
batch 70
batch 71
batch 72
batch 73
batch 74
batch 75
batch 76
batch 77
batch 78
batch 79
batch 80
batch 81
batch 82
batch 83
batch 84
batch 85
batch 86
batch 87
batch 88
batch 89
batch 90
batch 91
batch 92
batch 93
batch 94
batch 95
batch 96
batch 97
batch 98
batch 99
batch 100
batch 101
batch 102
batch 103
batch 104
batch 105
batch 106
batch 107
batch 108
batch 109
b

0